In [1]:
import pandas as pd

adult_df = pd.read_csv("adult_clean.csv")
print(adult_df.shape)

(30162, 15)


In [2]:
adult_df.head(2)

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K


In [3]:
print("--- CATEGORICAL VALUE COUNTS ---")

cat_cols = adult_df.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    print(f"\nDistribution for {col}:")
    # value_counts(dropna=False) includes missing values in the distribution
    print(adult_df[col].value_counts(dropna=False))

# 4. Continuous Distributions (Summary Statistics)
print("\n--- CONTINUOUS COLUMN SUMMARIES ---")
# .describe() computes count, mean, std, min, and quartiles for numeric columns
print(adult_df.describe())

--- CATEGORICAL VALUE COUNTS ---

Distribution for workclass:
workclass
Private             22286
Self-emp-not-inc     2499
Local-gov            2067
State-gov            1279
Self-emp-inc         1074
Federal-gov           943
Without-pay            14
Name: count, dtype: int64

Distribution for education:
education
HS-grad         9840
Some-college    6678
Bachelors       5044
Masters         1627
Assoc-voc       1307
11th            1048
Assoc-acdm      1008
10th             820
7th-8th          557
Prof-school      542
9th              455
12th             377
Doctorate        375
5th-6th          288
1st-4th          151
Preschool         45
Name: count, dtype: int64

Distribution for marital_status:
marital_status
Married-civ-spouse       14065
Never-married             9726
Divorced                  4214
Separated                  939
Widowed                    827
Married-spouse-absent      370
Married-AF-spouse           21
Name: count, dtype: int64

Distribution for occupatio

/tmp/ipykernel_38130/596291966.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = adult_df.select_dtypes(include=['object', 'category']).columns


In [4]:
import polars as pl
import opendp.prelude as dp

dp.enable_features("contrib")

## Statistical Query: Count

In [5]:
income_count = (adult_df["income"] == ">50K").sum()
print(income_count)

7508


In [11]:
context = dp.Context.compositor(
    data=pl.scan_csv(
        "adult_clean.csv",
        ignore_errors=True,
    ),
    privacy_unit=dp.unit_of(contributions=1),
    privacy_loss=dp.loss_of(epsilon=1.0),
    split_evenly_over=1,
)

query = context.query().filter(pl.col("income") == ">50K").select(dp.len())
income_count_dp = query.release().collect().item()

summary = query.summarize(alpha=0.05)

# column = summary["column"][0]
# aggregate = summary["aggregate"][0]
# distribution = summary["distribution"][0]
# scale = summary["scale"][0]
accuracy = summary["accuracy"][0]

print(income_count_dp)
print(accuracy)

7507
3.3756177665957137


## Statistical Query: Mean

In [31]:
average_age = adult_df["age"].mean()
print(average_age)
sum_age = adult_df["age"].sum()
print(sum_age)
len_age = len(adult_df["age"])
print(len_age)
print(sum_age / len_age)

38.437901995888865
1159364
30162
38.437901995888865


In [33]:
context = dp.Context.compositor(
    data=pl.scan_csv(
        "adult_clean.csv",
        ignore_errors=True,
    ),
    privacy_unit=dp.unit_of(contributions=1),
    privacy_loss=dp.loss_of(epsilon=1.0),
    split_evenly_over=1,
    margins=[
        dp.polars.Margin(
            max_length=40000
        ),
    ],
)

query = context.query().select(pl.col.age.cast(int).dp.sum(bounds=(18, 100)), dp.len())
average_age_dp = query.release().collect().with_columns(mean=pl.col.age / pl.col.len)
print(average_age_dp["age"].item())
print(average_age_dp["len"].item())
print(average_age_dp["mean"].item())

summary = query.summarize(alpha=0.05)

# column = summary["column"][0]
# aggregate = summary["aggregate"][0]
# distribution = summary["distribution"][0]
scale = summary["scale"][0]
accuracy = summary["accuracy"][0]

print(average_age_dp)
print(scale)
print(accuracy)

1159953
30162
38.45742987865526
shape: (1, 3)
┌─────────┬───────┬──────────┐
│ age     ┆ len   ┆ mean     │
│ ---     ┆ ---   ┆ ---      │
│ i64     ┆ u32   ┆ f64      │
╞═════════╪═══════╪══════════╡
│ 1159953 ┆ 30162 ┆ 38.45743 │
└─────────┴───────┴──────────┘
200.0
599.6458297114493


## Statistical Query: Quantile

In [7]:
median_hours = adult_df["hours_per_week"].median()
print(median_hours)
median_hours_q = adult_df["hours_per_week"].quantile(0.5)
print(median_hours_q)

40.0
40.0


## Statistical Query: Histogram

In [8]:
histogram = adult_df["relationship"].value_counts().sort_index()
print(histogram)

relationship
Husband           12463
Not-in-family      7726
Other-relative      889
Own-child          4466
Unmarried          3212
Wife               1406
Name: count, dtype: int64


## Candidate Selection: Best Candidate

In [10]:
education_hours = (
    adult_df.groupby("education")["hours_per_week"]
      .mean()
      .sort_values(ascending=False)
)

best_education = education_hours.head(5)

print("Education level with the highest average working hours:")
print(best_education)

Education level with the highest average working hours:
education
Prof-school    47.963100
Doctorate      47.832000
Masters        44.240934
Bachelors      42.948454
Assoc-voc      41.954093
Name: hours_per_week, dtype: float64


## Candidate Selection: Top-K

In [11]:
adult_df["high_income"] = (adult_df["income"] == ">50K").astype(int)

occupation_income = (
    adult_df.groupby("occupation")["high_income"]
      .mean()                # proportion earning >50K
      .sort_values(ascending=False)
)

top5_occupations = occupation_income.head(5)

print("\nTop 5 occupations with the highest proportion of high-income earners:")
print(top5_occupations)


Top 5 occupations with the highest proportion of high-income earners:
occupation
Exec-managerial    0.485220
Prof-specialty     0.448489
Protective-serv    0.326087
Tech-support       0.304825
Sales              0.270647
Name: high_income, dtype: float64


## Composition: Non-adaptive

## Composition: Adaptive

## ML: Regression

In [3]:
import opendp.prelude as dp
try:
   import sklearn
except ModuleNotFoundError:
    import pytest
    pytest.skip('Requires extra install')
dp.enable_features("contrib", "honest-but-curious", "idealized-numerics")
lin_reg = dp.sklearn.linear_model.LinearRegression(
    dp.max_divergence(),
    x_bounds=[(0, 10)],
    y_bounds=(0, 10),
    scale=1,
).fit(
    X=[[1,4], [2,5], [3,9], [4,1], [5,7]],
    y=[1, 2, 3, 4, 5],
)
lin_reg.predict([[10]])

array([9.6969697])

In [8]:
import pandas as pd

# Dataset URL
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"

columns = [
    "mpg",
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
    "origin"
]

# Read only the first 8 columns
df = pd.read_csv(
    url,
    sep=r"\s+",
    names=columns,
    usecols=range(8),
    na_values="?",
    engine="python"
)

# Convert horsepower to numeric
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")

# Remove rows with missing values
df = df.dropna().reset_index(drop=True)

# Save cleaned dataset
df.to_csv("auto_mpg_clean.csv", index=False)